# Himawari-9 ingest + processing on Colab

Runs the **bronze → silver** Himawari steps of `stormcast` close to the data (Google ↔ AWS is fast),
and ships only the small regridded output back to Google Drive, one zip per day.

**Before running:** upload `dist/stormcast-code.zip` (from the project folder on your PC) to
`MyDrive/stormcast/` on Google Drive.

The loop is resumable: if Colab disconnects, run all cells again and finished days are skipped.

**Afterwards on your PC:** download the `MyDrive/stormcast/himawari_days/` zips and extract each one into
`D:\hcmc-storm-nowcasting\data\` (they contain `silver/...` and `catalog/...`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/stormcast'
START, END = '2025-08-01', '2025-08-31'

In [ ]:
%%bash -s "$DRIVE_DIR"
set -e
rm -rf /content/stormcast && mkdir -p /content/stormcast
unzip -q "$1/stormcast-code.zip" -d /content/stormcast
cd /content/stormcast && pip install -q -e .

In [ ]:
import os, sys, time, shutil, zipfile, logging
from dataclasses import replace
from pathlib import Path
import pandas as pd

os.chdir('/content/stormcast')
sys.path.insert(0, '/content/stormcast/src')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)

from stormcast import pipeline
from stormcast.config import load_config

cfg = load_config('configs/default.yaml')
cfg = replace(
    cfg,
    paths=replace(cfg.paths, data_root='/content/data'),
    himawari=replace(cfg.himawari, keep_raw=False, download_workers=32),
    processing=replace(cfg.processing, workers=os.cpu_count()),
)
OUT = Path(DRIVE_DIR) / 'himawari_days'
OUT.mkdir(parents=True, exist_ok=True)
from stormcast.ingest.himawari import wanted_segments
print('segments:', wanted_segments(cfg), '| bands:', cfg.himawari.bands, '| cpus:', os.cpu_count())

In [ ]:
def ship_day(day):
    """Zip one day's silver files + catalog partitions (paths relative to data_root) to Drive."""
    root = cfg.paths.root
    files = list((cfg.paths.silver / 'himawari' / str(day)).glob('*.nc'))
    files += [cfg.paths.catalog / t / f'date={day}.parquet' for t in ('himawari_raw', 'himawari_frames')]
    tmp = Path('/content') / f'himawari_{day}.zip'
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_STORED) as z:  # NetCDF is already compressed
        for f in files:
            if f.exists():
                z.write(f, f.relative_to(root).as_posix())
    final = OUT / tmp.name
    shutil.move(str(tmp), str(final))  # copy to Drive, then drop the local zip
    return final

timings = []
for day in pd.date_range(START, END).date:
    if (OUT / f'himawari_{day}.zip').exists():
        print(day, 'already on Drive, skipped')
        continue
    t0 = time.time(); raw = pipeline.himawari_download(cfg, day)
    t1 = time.time(); frames = pipeline.himawari_process(cfg, day)
    t2 = time.time(); ship_day(day)
    row = {
        'day': str(day),
        'files': len(raw),
        'gb': raw['size_bytes'].sum() / 1e9 if len(raw) else 0,
        'download_s': round(t1 - t0), 'process_s': round(t2 - t1),
        'frames_ok': int((frames.get('status') == 'ok').sum()) if len(frames) else 0,
        'frames_total': len(frames),
    }
    timings.append(row)
    print(row)
    pd.DataFrame(timings).to_csv(OUT / f'timings_{START}_{END}.csv', mode='a',
                                 header=not (OUT / f'timings_{START}_{END}.csv').exists(), index=False)
    timings.clear()

In [ ]:
# Summary of everything shipped so far (numbers for the README / CV)
t = pd.read_csv(OUT / f'timings_{START}_{END}.csv')
print(t.tail())
print('days:', len(t), '| GB downloaded:', round(t.gb.sum(), 1),
      '| frames ok:', t.frames_ok.sum(), '/', t.frames_total.sum(),
      '| median download+process per day (min):', round((t.download_s + t.process_s).median() / 60, 1))